# MPECSS - MPECLib Benchmark

This notebook runs the **MPECLib** benchmark suite.

- **Dataset**: MPECLib
- **Problems**: 92
- **Estimated time**: 4-6 hours
- **Resume support**: Built-in

Run the cells in order. After a Kaggle restart, re-run cells 1-3, then use the **Resume** cell.

## 1. Configuration

In [ ]:
# MPECLib Configuration
import os
from pathlib import Path

DATASET = 'mpeclib'
RUN_TAG = 'Kaggle_MPECLib'

WORKERS = 4
TIMEOUT = 3600
NUM_PROBLEM = None  # None = all problems; set 5/10/etc for quick tests
SAVE_LOGS = True
SHUFFLE = True

# Repository (code)
REPO_DIR = "/kaggle/working/mpecss-kaggle"
REPO_GIT_URL = "https://github.com/mpecssalgorithm/mpecss.git"

# IMPORTANT: Results go directly to /kaggle/working/outputs to persist after session ends
OUTPUT_DIR = "/kaggle/working/outputs"

DATASET_JSON_DIR = "mpeclib-json"
EXPECTED_PROBLEM_COUNT = 92
REPO_DATASET_PATH = os.path.join(REPO_DIR, "benchmarks", DATASET, DATASET_JSON_DIR)
KAGGLE_INPUT_ROOT = Path('/kaggle/input')


def _count_json_files(path):
    path = Path(path)
    if not path.is_dir():
        return 0
    return sum(1 for item in path.iterdir() if item.is_file() and item.name.endswith('.json'))


def _resolve_dataset_path():
    checked = []
    seen = set()
    candidates = [Path(REPO_DATASET_PATH)]

    if KAGGLE_INPUT_ROOT.exists():
        patterns = [
            f'benchmarks/{DATASET}/{DATASET_JSON_DIR}',
            f'*/benchmarks/{DATASET}/{DATASET_JSON_DIR}',
            f'*/{DATASET}/{DATASET_JSON_DIR}',
            DATASET_JSON_DIR,
            f'*/{DATASET_JSON_DIR}',
            f'*/{DATASET}',
            f'**/{DATASET_JSON_DIR}',
        ]
        for pattern in patterns:
            candidates.extend(sorted(KAGGLE_INPUT_ROOT.glob(pattern)))

    for candidate in candidates:
        possible_paths = [candidate]
        if candidate.is_dir() and candidate.name != DATASET_JSON_DIR:
            possible_paths.append(candidate / DATASET_JSON_DIR)

        for possible in possible_paths:
            possible = possible.resolve()
            if possible in seen:
                continue
            seen.add(possible)
            checked.append(str(possible))
            if _count_json_files(possible) > 0:
                return str(possible)

    return str(REPO_DATASET_PATH)


DATASET_PATH = _resolve_dataset_path()
print(f"[INFO] Benchmark path: {DATASET_PATH}")


## 2. Clone Repository

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_DIR = Path(REPO_DIR)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

REPO_COMMIT = "main"
print(f"Cloning: {REPO_GIT_URL} @ {REPO_COMMIT}")
subprocess.run(["git", "clone", "--depth", "1", REPO_GIT_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_COMMIT], check=True, cwd=str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))
print(f"[OK] Repository ready at: {REPO_DIR}")

## 3. Install Dependencies

In [ ]:
%%bash
cd /kaggle/working/mpecss-kaggle
pip install -q --upgrade pip
pip install -q -e .
echo "[OK] Installation complete"

## 4. Verify Setup

In [ ]:
import os

DATASET_PATH = _resolve_dataset_path()
problem_count = _count_json_files(DATASET_PATH)

print(f"Dataset path: {DATASET_PATH}")

if problem_count == 0:
    raise RuntimeError(
        f"[FATAL] Dataset not found. Add a Kaggle Input dataset containing "
        f"benchmarks/{DATASET}/{DATASET_JSON_DIR}/*.json or a {DATASET_JSON_DIR} folder."
    )

if problem_count != EXPECTED_PROBLEM_COUNT:
    print(f"[WARN] Expected {EXPECTED_PROBLEM_COUNT} {DATASET} problems, found {problem_count}")
print(f"[OK] Found {problem_count} problems")


## 5. System Info

In [ ]:
import multiprocessing
import platform
import psutil

print("=" * 50)
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"CPUs: {multiprocessing.cpu_count()}")
mem = psutil.virtual_memory()
print(f"RAM: {mem.available / 1024**3:.1f} GB available / {mem.total / 1024**3:.1f} GB total")
print("=" * 50)

## 6. Preflight Checks

In [ ]:
%%bash
cd /kaggle/working/mpecss-kaggle
python mpecss/helpers/preflight_checks.py

## 7. Define Runner

In [ ]:
import subprocess
import sys


def run_benchmark(resume=False, summary=False, retry_failed=False):
    cmd = [
        sys.executable,
        f"{str(REPO_DIR)}/kaggle_setup/resumable_benchmark.py",
        "--dataset", DATASET,
        "--repo-dir", str(REPO_DIR),
        "--tag", RUN_TAG,
        "--workers", str(WORKERS),
        "--timeout", str(TIMEOUT),
        "--path", str(DATASET_PATH),
        "--output-dir", str(OUTPUT_DIR),  # CRITICAL: Save to persistent location
    ]
    if NUM_PROBLEM is not None:
        cmd.extend(["--num-problems", str(NUM_PROBLEM)])
    if SAVE_LOGS:
        cmd.append("--save-logs")
    if SHUFFLE:
        cmd.append("--shuffle")
    if resume or retry_failed:
        cmd.append("--resume-latest")
    if retry_failed:
        cmd.append("--retry-failed")
    if summary:
        cmd.append("--summary-only")

    print("+ " + " \".join(str(x) for x in cmd))
    subprocess.run(cmd)


## 8. Run Benchmark (Fresh Start)

In [ ]:
run_benchmark()

## 9. Resume (Manual, After Restart)


In [ ]:
# run_benchmark(resume=True)


## 10. Retry Only Failed Problems (Manual)


In [ ]:
# run_benchmark(resume=True, retry_failed=True)


## 11. Progress Summary


In [ ]:
run_benchmark(summary=True)

## 12. Copy Results


In [ ]:
from pathlib import Path
import shutil

output_dir = Path(OUTPUT_DIR)
if not output_dir.exists():
    print(f"[WARN] Output directory not found: {output_dir}")
else:
    archive = shutil.make_archive(str(output_dir), "zip", root_dir=str(output_dir))
    print(f"[OK] Results directory: {output_dir}")
    print(f"[OK] Download archive: {archive}")
    for item in sorted(output_dir.iterdir()):
        print(item)


## 13. Software Versions


In [ ]:
import casadi, numpy, scipy, pandas, platform
print(f"Python: {platform.python_version()}")
print(f"NumPy: {numpy.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Pandas: {pandas.__version__}")
print(f"CasADi: {casadi.__version__}")